# 1. Word Embeddings

**What this notebook teaches.** A word embedding maps every word to a vector such that words used in similar contexts get similar vectors. We load pretrained [fastText](https://fasttext.cc/docs/en/crawl-vectors.html) vectors (300 dimensions, trained on Common Crawl) and explore the geometry of the embedding space:

* nearest neighbours of a word are semantically related words;
* vector arithmetic captures relations: `king - man + woman ≈ queen`;
* analogies ("berlin is to germany as ? is to france") are solved the same way.

The next notebooks use embeddings as the first layer of a text classifier.

In [ ]:
# download the fasttext embeddings (only the first time).
# WARNING: this downloads cc.en.300.bin.gz (~4.5 GB) into the current folder
# and unzips it to cc.en.300.bin (~7 GB); loading it needs ~8 GB of RAM.
import fasttext.util
fasttext.util.download_model('en', if_exists='ignore')  # English

In [ ]:
import fasttext
import numpy as np

In [ ]:
model = fasttext.load_model('cc.en.300.bin')

In [ ]:
king_emb = model.get_word_vector('King')
king_emb.shape, king_emb

## Vector Operations on Representations

In [ ]:
man_emb = model.get_word_vector('Man')
woman_emb = model.get_word_vector('Woman')

In [ ]:
new_emb = king_emb - man_emb + woman_emb

`get_nearest_neighbor_given_emb` searches only the 10,000 most frequent words and uses the Euclidean distance; the words of the query themselves (`King`, `Woman`) are allowed as answers. fastText's own `get_nearest_neighbors` and `get_analogies` below use the cosine similarity and exclude the query words.

In [ ]:
def get_nearest_neighbor_given_emb(model, emb, k=10):
    embs = []
    for word in model.words[:10000]:
        embs.append(model.get_word_vector(word))
    embs = np.array(embs)
    nn_token_ids = ((embs - emb) ** 2).sum(-1).argsort()[:k]
    return [model.words[nn_token_id] for nn_token_id in nn_token_ids]

get_nearest_neighbor_given_emb(model, new_emb, k=1)

## Embedding Space

In [ ]:
model.get_nearest_neighbors('Bangalore', k=5)

## Analogy

In [ ]:
model.get_analogies("berlin", "germany", "france", k=1)

In [ ]:
model.get_analogies("Sony", "Playstation", "Xbox", k=3)